In [0]:
import pydicom
from tqdm import tqdm
import matplotlib.pyplot as plt
import numpy as np
import time
import copy
import os
from io import BytesIO
from pydicom.filebase import DicomFileLike
from openai import OpenAI
from PIL import Image
import base64



In [0]:
input_dir = "/Volumes/1_inland/evan_demo/misc/20260304_162300/"


def list_dcm_files(folder):
    files = os.listdir(folder)
    files = [x for x in files if x.endswith(".dcm")  and "DICOMDIR" not in x]
    return files

df = (
    spark.read.format("binaryFile")
    .option("pathGlobFilter", "*.dcm")
    .option("recursiveFileLookup", "true")
    .load(input_dir)
)
print(df.count())
display(df)

In [0]:
import pandas as pd
from pathlib import Path

def llm_pii_detection_batch(pdf_iter):
    system_prompt = "Output the result in a structured format with 3 required fields: is_text_detected, is_personal_information_detected, and confidence_score. Valid values for is_text_detected are yes, no, and not_sure. Valid values for is_personal_information_detected are yes, no, and not_sure. confidence_score is an integer from 0 to 100. 0 indicates high confidence of no text detected. 100 indicates high confidence of text being detected. 50 indicates uncertain. Do not leave any field empty. Always respond with 3 fields everytime."

    user_prompt = "Tell me if there is any text in the image. Also tell me if these texts contain any personal information e.g. name, date of birth, scan date, ID numbers. Then give me a confidence score."

    deployment_name = "gpt-4.1-mini"

    endpoint = "https://telefonica-hackathon-2026.cognitiveservices.azure.com/openai/v1/"
    api_key = ""

    client = OpenAI(base_url=f"{endpoint}",api_key=api_key)

    for pdf in pdf_iter:
        redacted_rows = []
        for path, content in zip(pdf['path'], pdf['content']):
            try:
                is_loaded_ok = False

                ds = pydicom.dcmread(BytesIO(content), force=True)

                #ds.decompress()


                # Extract pixel array
                pixel_array = ds.pixel_array.astype(np.float32)

                # Normalize to 0-255
                pixel_array -= pixel_array.min()
                pixel_array /= pixel_array.max()
                pixel_array *= 255.0
                pixel_array = pixel_array.astype(np.uint8)

                # Convert to PIL image
                image = Image.fromarray(pixel_array)

                # Save PNG to memory buffer
                buffer = BytesIO()
                image.save(buffer, format="PNG")

                # Convert to Base64
                base64_str = base64.b64encode(buffer.getvalue()).decode("utf-8")


                is_loaded_ok = True




                completion = client.chat.completions.create(
                    model=deployment_name,
                    messages=[
                        {
                            "role": "system",
                            "content": system_prompt,
                        },
                        {
                            "role": "user",
                            "content": [
                                {
                                    "type": "text", "text": user_prompt
                                },
                                {
                                    "type": "image_url",
                                    "image_url": {"url": f"data:image/png;base64,{base64_str}"}
                                }

                            ]
                        }
                    ]
                )

                result = completion.choices[0].message.content
                
                redacted_rows.append((path, is_loaded_ok, result, None))
            except Exception as e:
                redacted_rows.append((path, is_loaded_ok, None, str(e)[:1000]))


        yield pd.DataFrame(redacted_rows, columns=["input_path", "is_loaded_ok", "llm_result", "error_message"])

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, BinaryType, BooleanType

# Define output schema: path + redacted DICOM binary
output_schema = StructType([
    StructField("input_path", StringType(), True),
    StructField("is_loaded_ok", BooleanType(), True),
    StructField("llm_result", StringType(), True),
    StructField("error_message", StringType(), True)
])

# Run mapInPandas redaction
pii_detect_df = df.mapInPandas( llm_pii_detection_batch, schema=output_schema)
# Show saved paths
results = pii_detect_df.collect()

In [0]:
display(results)